In [12]:
# !pip install ISLP

In [77]:
import os
import pandas as pd
import numpy as np 

from xgboost import XGBRegressor  

from sklearn.model_selection import KFold ,cross_val_score, cross_val_predict
from tqdm import tqdm 
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")


In [78]:
from ISLP import load_data
Carseats = load_data('Carseats')
X , y = Carseats.drop('Sales' , axis = 1) , Carseats['Sales']

# Carseats.columns
X_train , X_test , y_train , y_test = train_test_split(X , y, random_state=26 , test_size=0.2, stratify=Carseats['ShelveLoc'])

In [79]:
Carseats.describe().iloc[:,:4]

,Sales,CompPrice,Income,Advertising
count,400.000000,400.000000,400.000000,400.000000
mean,7.496325,124.975000,68.657500,6.635000
std,2.824115,15.334512,27.986037,6.650364
min,0.000000,77.000000,21.000000,0.000000
25%,5.390000,115.000000,42.750000,0.000000
50%,7.490000,125.000000,69.000000,5.000000
75%,9.320000,135.000000,91.000000,12.000000
max,16.270000,175.000000,120.000000,29.000000


In [80]:
Carseats

,Sales,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US
0,9.50,138,73,11,276,120,Bad,42,17,Yes,Yes
1,11.22,111,48,16,260,83,Good,65,10,Yes,Yes
2,10.06,113,35,10,269,80,Medium,59,12,Yes,Yes
3,7.40,117,100,4,466,97,Medium,55,14,Yes,Yes
4,4.15,141,64,3,340,128,Bad,38,13,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...
395,12.57,138,108,17,203,128,Good,33,14,Yes,Yes
396,6.14,139,23,3,37,120,Medium,55,11,No,Yes
397,7.41,162,26,12,368,159,Medium,40,18,Yes,Yes
398,5.94,100,79,7,284,95,Bad,50,12,Yes,Yes


In [81]:
Carseats.dtypes

Sales           float64
CompPrice         int64
Income            int64
Advertising       int64
Population        int64
Price             int64
ShelveLoc      category
Age               int64
Education         int64
Urban          category
US             category
dtype: object

In [82]:
ohc = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
ct = make_column_transformer(
                            (ohc,           make_column_selector(dtype_exclude = ['int64','float64'])),
                            ("passthrough", make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False).set_output(transform='pandas') 

X_ohe_train = ct.fit_transform(X_train)
X_ohe_test = ct.transform(X_test)


In [83]:
shelves=[['Bad',"Medium","Good"]]
ord_enc = OrdinalEncoder(categories=shelves)

ohc = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')

ct = make_column_transformer(
                            (ohc,          ['Urban' ,'US']),
                            (ord_enc, ['ShelveLoc']),
                            ("passthrough",make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out = False).set_output(transform='pandas') 

# ct_ohc_train = ct.fit_transform(X_train)
# ct_ohc_test = ct.transform(X_test)

In [84]:
gbm = XGBRegressor(random_state=26) 
pipe = Pipeline([('OHE',ct), ('GBM', gbm)])
# pipe.get_params()

params = {
    "GBM__learning_rate":np.linspace(0.01,0.9,10),
    "GBM__max_depth":[None,2,3,4],
    "GBM__n_estimators":[25,50,100]
}
gcv = GridSearchCV(pipe,param_grid = params , cv = kfold, scoring='r2')
gcv.fit(X_train, y_train)
# pipe.get_params()
print(f'Best Params : {gcv.best_params_}')
print(f'Best Score : {gcv.best_score_}')

y_pred = gcv.predict(X_test)
print(r2_score(y_test, y_pred))

Best Params : {'GBM__learning_rate': np.float64(0.20777777777777778), 'GBM__max_depth': 2, 'GBM__n_estimators': 100}
Best Score : 0.7564564206569815
0.786994846981635


In [85]:
lgm = LGBMRegressor(random_state=26,verbose=-1)
pipe = Pipeline([('OHE',ct), ('LGBM', lgm)])

kfold = KFold(random_state=26, n_splits=5, shuffle=True)
params = {
    "LGBM__learning_rate":np.linspace(0.01,0.9,10),
    "LGBM__max_depth":[None,2,3,4],
    "LGBM__n_estimators":[25,50,100]
}

gcv = GridSearchCV(pipe,param_grid = params , cv = kfold, scoring='r2')

gcv.fit(X_train, y_train)
print(f'Best Params : {gcv.best_params_}')
print(f'Best Score : {gcv.best_score_}')

y_pred = gcv.predict(X_test)
print(r2_score(y_test, y_pred))

Best Params : {'LGBM__learning_rate': np.float64(0.30666666666666664), 'LGBM__max_depth': 2, 'LGBM__n_estimators': 50}
Best Score : 0.748544303635134
0.7997925183742228


In [86]:
rfor = RandomForestRegressor(random_state=26 )
pipel = Pipeline([("CT",ct ),("RF",rfor)])
pipel.get_params()
kfold = KFold(random_state=26, n_splits=5, shuffle=True)
params = {
    'RF__max_features' : [2,3,4,5],
    "RF__n_estimators":[10,25,50,100]
}

gcv = GridSearchCV(pipel,param_grid = params , cv = kfold, scoring='r2', n_jobs=-1)

gcv.fit(X_train, y_train)
print(f'Best Params : {gcv.best_params_}')
print(f'Best Score : {gcv.best_score_}')

y_pred = gcv.predict(X_test)
print(r2_score(y_test, y_pred))

Best Params : {'RF__max_features': 5, 'RF__n_estimators': 100}
Best Score : 0.6658263522475494
0.647479851449854


# Randoimized Search CV

In [88]:
from sklearn.model_selection import RandomizedSearchCV

In [93]:


gbm = XGBRegressor(random_state=26) 
pipe = Pipeline([('OHE',ct), ('GBM', gbm)])
 
# pipe.get_params()

params = {
    "GBM__learning_rate":np.linspace(0.01,0.9,10),
    "GBM__max_depth":[None,2,3,4],
    "GBM__n_estimators":[25,50,100]
}

rgcv = RandomizedSearchCV(pipe , param_distributions = params , n_iter = 20,cv = kfold, scoring='r2', n_jobs=-1)

rgcv.fit(X_train, y_train)
print(f'Best Params : {rgcv.best_params_}')
print(f'Best Score : {rgcv.best_score_}')

y_pred = rgcv.predict(X_test)
print(r2_score(y_test, y_pred))

Best Params : {'GBM__n_estimators': 100, 'GBM__max_depth': 2, 'GBM__learning_rate': np.float64(0.20777777777777778)}
Best Score : 0.7564564206569815
0.786994846981635


In [94]:
bm_xgb = rgcv.best_estimator_
y_pred = bm_xgb.predict(X_test)
print(r2_score(y_test, y_pred))

0.786994846981635
